# Analisis Spatio-Temporal Gempa Bumi Indonesia
### Notebook 02 — Tren Temporal, Dekomposisi Musiman, KDE Spasial, Gutenberg-Richter, Seismic Gap & Skor Bahaya

## Cell 1 — Load Data & Distribusi Zona

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import gaussian_kde
from statsmodels.tsa.seasonal import seasonal_decompose

# ── Paths ──────────────────────────────────────────────────────────────────
WORKDIR = Path(".")
INPUT_CSV = WORKDIR / "indonesia_earthquakes_clustered.csv"
OUTDIR    = WORKDIR / "output_spatio_temporal"
OUTDIR.mkdir(exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
df["time"] = pd.to_datetime(df["time"], format="mixed", errors="coerce")
df["year"] = df["time"].dt.year

ZONES = ["Zona Sumatera", "Zona Jawa-Bali-NTB", "Zona Sulawesi-NTT", "Zona Maluku", "Zona Papua"]
COLORS = {
    "Zona Sumatera":      "#e41a1c",
    "Zona Jawa-Bali-NTB": "#377eb8",
    "Zona Sulawesi-NTT":  "#4daf4a",
    "Zona Maluku":        "#ff7f00",
    "Zona Papua":         "#984ea3",
}

print(f"Dataset dimuat: {len(df):,} baris x {len(df.columns)} kolom")
print(f"Rentang waktu : {df['time'].min().date()} - {df['time'].max().date()}")
print(f"Kolom         : {list(df.columns)}")
print()
print("=" * 55)
print(f"{'Zona':<25} {'N Events':>9} {'%':>8}")
print("-" * 55)
for z in ZONES:
    n = (df["zone_name"] == z).sum()
    print(f"  {z:<23} {n:>9,} {n/len(df)*100:>7.1f}%")
print("=" * 55)
print(f"  {'TOTAL':<23} {len(df):>9,} {'100.0%':>8}")


## Cell 2 — Tren Temporal per Zona

In [ ]:
# ── Hitung event per tahun per zona ───────────────────────────────────────
yearly = (df.groupby(["year", "zone_name"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=ZONES, fill_value=0))

# Hanya tahun 1963–2024 (data lengkap per tahun)
yearly = yearly.loc[(yearly.index >= 1963) & (yearly.index <= 2024)]

# Perubahan per-dekade (%)
decades = [1970, 1980, 1990, 2000, 2010, 2020]
decade_change = {}
for z in ZONES:
    chg = {}
    for d in decades[1:]:
        prev = yearly.loc[(yearly.index >= d-10) & (yearly.index < d), z].mean()
        curr = yearly.loc[(yearly.index >= d) & (yearly.index < d+10), z].mean()
        chg[d] = (curr - prev) / prev * 100 if prev > 0 else 0
    decade_change[z] = chg

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle("Tren Temporal Aktivitas Seismik per Zona (1963–2024)",
             fontsize=14, fontweight="bold", y=0.98)

# Panel 1: line chart event/tahun
ax1 = axes[0]
for z in ZONES:
    ax1.plot(yearly.index, yearly[z], color=COLORS[z], lw=1.5, label=z, alpha=0.85)
    # trendline
    coeffs = np.polyfit(yearly.index, yearly[z], 1)
    trend  = np.polyval(coeffs, yearly.index)
    ax1.plot(yearly.index, trend, color=COLORS[z], lw=0.8, ls="--", alpha=0.5)
ax1.set_ylabel("Jumlah Event / Tahun")
ax1.set_title("Jumlah Event per Tahun per Zona (garis putus = tren linier)", fontsize=11)
ax1.legend(loc="upper left", fontsize=8, ncol=2)
ax1.grid(axis="y", alpha=0.3)

# Panel 2: stacked bar per dekade
ax2 = axes[1]
dec_labels = ["1960s","1970s","1980s","1990s","2000s","2010s","2020–24"]
dec_ranges  = [(1963,1969),(1970,1979),(1980,1989),(1990,1999),(2000,2009),(2010,2019),(2020,2024)]
dec_totals  = {z: [] for z in ZONES}
for s, e in dec_ranges:
    for z in ZONES:
        dec_totals[z].append(yearly.loc[(yearly.index>=s)&(yearly.index<=e), z].sum())

x = np.arange(len(dec_labels))
bottom = np.zeros(len(dec_labels))
for z in ZONES:
    vals = np.array(dec_totals[z])
    ax2.bar(x, vals, bottom=bottom, color=COLORS[z], label=z, width=0.65, alpha=0.88)
    bottom += vals
ax2.set_xticks(x)
ax2.set_xticklabels(dec_labels)
ax2.set_ylabel("Total Event")
ax2.set_title("Total Event per Dekade (stacked)", fontsize=11)
ax2.legend(loc="upper left", fontsize=8, ncol=2)
ax2.grid(axis="y", alpha=0.3)

# Panel 3: heatmap rata-rata event/tahun per dekade × zona
ax3 = axes[2]
dec_mean = pd.DataFrame(index=dec_labels, columns=ZONES, dtype=float)
for idx, (s, e) in enumerate(dec_ranges):
    n_yrs = e - s + 1
    for z in ZONES:
        dec_mean.loc[dec_labels[idx], z] = dec_totals[z][idx] / n_yrs

im = ax3.imshow(dec_mean.values.astype(float), aspect="auto", cmap="YlOrRd")
ax3.set_xticks(range(len(ZONES)))
ax3.set_xticklabels([z.replace("Zona ","") for z in ZONES], rotation=15, ha="right", fontsize=9)
ax3.set_yticks(range(len(dec_labels)))
ax3.set_yticklabels(dec_labels, fontsize=9)
ax3.set_title("Rata-rata Event / Tahun per Dekade × Zona (heatmap)", fontsize=11)
plt.colorbar(im, ax=ax3, label="Event/tahun")
for i in range(len(dec_labels)):
    for j in range(len(ZONES)):
        val = dec_mean.iloc[i, j]
        ax3.text(j, i, f"{val:.0f}", ha="center", va="center",
                 fontsize=7, color="black" if val < dec_mean.values.max()*0.6 else "white")

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = OUTDIR / "temporal_trend_per_zona.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n[OK] Disimpan: {out_path.name}")

# Ringkasan tren
print("\nTren linier (koef arah event/tahun):")
print(f"{'Zona':<25} {'Slope':>10} {'Interpretasi':>15}")
print("-" * 55)
for z in ZONES:
    slope = np.polyfit(yearly.index, yearly[z], 1)[0]
    interp = "Naik" if slope > 0.5 else ("Turun" if slope < -0.5 else "Stabil")
    print(f"  {z:<23} {slope:>+9.2f}  {interp:>15}")


## Cell 3 — Dekomposisi Musiman per Zona

In [ ]:
# ── Buat time series bulanan per zona (2000–2023 agar ada 2 × 12 periode) ──
df_full = df[(df["year"] >= 2000) & (df["year"] <= 2023)].copy()
df_full["year_month"] = df_full["time"].dt.to_period("M")

monthly_zona = (df_full.groupby(["year_month", "zone_name"])
                       .size()
                       .unstack(fill_value=0)
                       .reindex(columns=ZONES, fill_value=0))
monthly_zona.index = monthly_zona.index.to_timestamp()

print(f"Data bulanan: {len(monthly_zona)} bulan ({monthly_zona.index.min().date()} – {monthly_zona.index.max().date()})")

# ── Dekomposisi & Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(len(ZONES), 4, figsize=(18, 3.5 * len(ZONES)))
fig.suptitle("Dekomposisi Musiman (Additive, period=12) — Event/Bulan per Zona (2000–2023)",
             fontsize=13, fontweight="bold", y=1.005)

decomp_results = {}
for row, z in enumerate(ZONES):
    ts = monthly_zona[z].astype(float)
    result = seasonal_decompose(ts, model="additive", period=12, extrapolate_trend="freq")
    decomp_results[z] = result

    comps = [("Observed",   ts),
             ("Trend",      result.trend),
             ("Seasonal",   result.seasonal),
             ("Residual",   result.resid)]

    for col, (name, series) in enumerate(comps):
        ax = axes[row, col]
        color = COLORS[z] if name != "Residual" else "gray"
        ls    = "-" if name != "Residual" else "."
        ax.plot(series.index, series.values, color=color, lw=0.9, marker=ls if name=="Residual" else None,
                markersize=2, alpha=0.85)
        if row == 0:
            ax.set_title(name, fontsize=10, fontweight="bold")
        if col == 0:
            ax.set_ylabel(z.replace("Zona ", ""), fontsize=8, rotation=90, labelpad=3)
        ax.grid(alpha=0.2)
        ax.tick_params(labelsize=7)
        if row < len(ZONES)-1:
            ax.set_xticklabels([])

plt.tight_layout()
out_path = OUTDIR / "seasonal_decompose_per_zona.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Disimpan: {out_path.name}")

# Ringkasan seasonal strength
print("\nKekuatan pola musiman per zona:")
print(f"{'Zona':<25} {'Var Seasonal':>14} {'Var Resid':>12} {'Seasonal Str':>14}")
print("-" * 68)
for z in ZONES:
    r   = decomp_results[z]
    vs  = np.nanvar(r.seasonal.values)
    vr  = np.nanvar(r.resid.values)
    ss  = vs / (vs + vr) if (vs + vr) > 0 else 0
    print(f"  {z:<23} {vs:>14.2f} {vr:>12.2f} {ss:>13.3f}")


## Cell 4 — Spatial KDE per Zona

In [ ]:
# ── KDE Spasial per Zona ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Distribusi Spasial & Hotspot KDE per Zona Seismik", fontsize=13, fontweight="bold")
axes_flat = axes.flatten()

kde_summary = {}
for idx, z in enumerate(ZONES):
    ax  = axes_flat[idx]
    sub = df[df["zone_name"] == z][["latitude", "longitude", "mag"]].dropna()

    lat = sub["latitude"].values
    lon = sub["longitude"].values

    # Bounding box
    lat_min, lat_max = lat.min() - 0.5, lat.max() + 0.5
    lon_min, lon_max = lon.min() - 0.5, lon.max() + 0.5

    # KDE
    xy     = np.vstack([lon, lat])
    kernel = gaussian_kde(xy, bw_method=0.15)
    xi     = np.linspace(lon_min, lon_max, 150)
    yi     = np.linspace(lat_min, lat_max, 150)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi     = kernel(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)

    # Plot
    cf = ax.contourf(Xi, Yi, Zi, levels=15, cmap="hot_r", alpha=0.75)
    ax.scatter(lon, lat, s=0.3, c=COLORS[z], alpha=0.2, rasterized=True)
    plt.colorbar(cf, ax=ax, shrink=0.8, label="Kepadatan KDE")

    # Hotspot: titik puncak KDE
    peak_idx = np.unravel_index(np.argmax(Zi), Zi.shape)
    peak_lon = xi[peak_idx[1]]
    peak_lat = yi[peak_idx[0]]
    ax.plot(peak_lon, peak_lat, "w*", markersize=12, markeredgecolor="black", zorder=5)

    ax.set_title(z, fontsize=10, color=COLORS[z], fontweight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.2)
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

    kde_summary[z] = {
        "n_events":   len(sub),
        "hotspot_lat": round(peak_lat, 2),
        "hotspot_lon": round(peak_lon, 2),
        "lat_range":  (round(lat_min+0.5,1), round(lat_max-0.5,1)),
        "lon_range":  (round(lon_min+0.5,1), round(lon_max-0.5,1)),
        "mean_mag":   round(sub["mag"].mean(), 2),
    }

# Sembunyikan subplot kosong (ke-6)
axes_flat[-1].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_path = OUTDIR / "spatial_per_zona.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Disimpan: {out_path.name}")

# Ringkasan
print("\nRingkasan KDE Spasial:")
print(f"{'Zona':<25} {'N':>7} {'Hotspot Lat':>12} {'Hotspot Lon':>12} {'Mean Mag':>10}")
print("-" * 75)
for z in ZONES:
    s = kde_summary[z]
    print(f"  {z:<23} {s['n_events']:>7,} {s['hotspot_lat']:>12.2f} {s['hotspot_lon']:>12.2f} {s['mean_mag']:>10.2f}")


## Cell 5 — Gutenberg-Richter b-Value, Recurrence Interval & Energy Release

In [ ]:
MC = 4.0   # magnitude completeness threshold
M_RECUR = 5.0  # magnitude untuk recurrence interval

# ── Hitung b-value (MLE) per zona ─────────────────────────────────────────
def compute_bvalue_mle(mags, mc=MC):
    mags_above = mags[mags >= mc]
    n = len(mags_above)
    if n < 10:
        return np.nan, np.nan, n
    mean_m = mags_above.mean()
    b = np.log10(np.e) / (mean_m - mc)
    sigma_b = b / np.sqrt(n)
    return round(b, 4), round(sigma_b, 4), n

# ── Recurrence interval (Poisson) ─────────────────────────────────────────
def recurrence_interval(mags, b, mc=MC, m_target=M_RECUR, n_years=62):
    if np.isnan(b): return np.nan
    a = np.log10(len(mags[mags >= mc]) / n_years) + b * mc
    annual_rate = 10 ** (a - b * m_target)
    return round(1 / annual_rate, 2) if annual_rate > 0 else np.nan

# ── Energy release (J) ────────────────────────────────────────────────────
def energy_joules(mag):
    return 10 ** (1.5 * mag + 4.8)

gr_stats = {}
for z in ZONES:
    mags = df[df["zone_name"] == z]["mag"].dropna().values
    b, sigma_b, n_above = compute_bvalue_mle(mags)
    ri  = recurrence_interval(mags, b)
    e_total = energy_joules(mags).sum()
    e_mean  = energy_joules(mags).mean()
    gr_stats[z] = {"b": b, "sigma_b": sigma_b, "n_above_mc": n_above,
                   "recurrence_yr": ri,
                   "energy_total_J": e_total, "energy_mean_J": e_mean}

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Gutenberg-Richter & Kurva FMD per Zona", fontsize=13, fontweight="bold")
axes_flat = axes.flatten()

mag_bins = np.arange(MC, 9.1, 0.2)

for idx, z in enumerate(ZONES):
    ax   = axes_flat[idx]
    mags = df[df["zone_name"] == z]["mag"].dropna().values
    b    = gr_stats[z]["b"]
    n_mc = gr_stats[z]["n_above_mc"]

    # FMD empiris (kumulatif)
    counts_cum = np.array([(mags >= m).sum() for m in mag_bins])
    ax.semilogy(mag_bins, counts_cum, "o", color=COLORS[z], ms=4, alpha=0.8, label="FMD empiris")

    # Garis regresi G-R
    if not np.isnan(b):
        log_n_mc = np.log10(n_mc)
        a_val    = log_n_mc + b * MC
        gr_line  = 10 ** (a_val - b * mag_bins)
        ax.semilogy(mag_bins, gr_line, "k--", lw=1.5, alpha=0.8, label=f"G-R fit (b={b:.3f})")

    ax.axvline(MC, color="gray", ls=":", lw=1, label=f"Mc={MC}")
    ax.set_title(z, fontsize=10, color=COLORS[z], fontweight="bold")
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("N kumulatif")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

axes_flat[-1].set_visible(False)
plt.tight_layout(rect=[0, 0, 1, 0.96])
out_path = OUTDIR / "bvalue_per_zona.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Disimpan: {out_path.name}")

# Tabel ringkasan
print("\nGutenberg-Richter & Recurrence Summary:")
print(f"{'Zona':<25} {'b-value':>9} {'±sigma':>8} {'N(≥Mc)':>9} "
      f"{'RI M≥5 (yr)':>13} {'E-total (J)':>14}")
print("-" * 85)
for z in ZONES:
    s = gr_stats[z]
    print(f"  {z:<23} {s['b']:>9.3f} {s['sigma_b']:>8.4f} {s['n_above_mc']:>9,} "
          f"{str(s['recurrence_yr']):>13} {s['energy_total_J']:>14.3e}")


## Cell 6 — Seismic Gap Detection

In [ ]:
import re

RECENT_START = 2016
RECENT_END   = 2026
HIST_END     = RECENT_START - 1

# ── Temporal Gap: zona dengan event/tahun di bawah mean - 1.5*sigma ──────
yearly_all = (df.groupby(["year","zone_name"]).size()
                .unstack(fill_value=0)
                .reindex(columns=ZONES, fill_value=0))
yearly_all = yearly_all.loc[(yearly_all.index >= 1963) & (yearly_all.index <= 2024)]

temporal_gap_zones = {}
for z in ZONES:
    series = yearly_all[z]
    mu, sigma = series.mean(), series.std()
    threshold = mu - 1.5 * sigma
    gap_years = series[series < threshold].index.tolist()
    temporal_gap_zones[z] = {"mean": mu, "std": sigma,
                              "threshold": threshold, "gap_years": gap_years}

print("Temporal Gap (event/tahun < mean - 1.5*sigma):")
print(f"{'Zona':<25} {'Mean':>7} {'Std':>7} {'Threshold':>11} {'Gap Years'}")
print("-" * 75)
for z in ZONES:
    g = temporal_gap_zones[z]
    gyr = str(g["gap_years"][:5]) + ("..." if len(g["gap_years"]) > 5 else "")
    print(f"  {z:<23} {g['mean']:>7.1f} {g['std']:>7.1f} {g['threshold']:>11.1f}  {gyr}")

# ── Spatial Gap: grid 1x1 deg yang kosong di RECENT tapi ada di HIST ──────
df_hist   = df[df["year"] <= HIST_END]
df_recent = df[(df["year"] >= RECENT_START) & (df["year"] <= RECENT_END)]

hist_grids   = set(df_hist["grid_id"].unique())
recent_grids = set(df_recent["grid_id"].unique())
spatial_gaps = hist_grids - recent_grids

def grid_to_latlon(gid):
    m = re.match(r"LAT(-?\d+)_LON(-?\d+)", gid)
    if not m:
        return np.nan, np.nan
    return int(m.group(1)) + 0.5, int(m.group(2)) + 0.5

gap_coords = [grid_to_latlon(g) for g in spatial_gaps]
gap_df = pd.DataFrame(gap_coords, columns=["lat_center","lon_center"])
gap_df["grid_id"] = list(spatial_gaps)

grid_zone = (df_hist[["grid_id","zone_name"]]
             .drop_duplicates(subset="grid_id", keep="first")
             .set_index("grid_id")["zone_name"])
gap_df["zone_name"] = gap_df["grid_id"].map(grid_zone)

print(f"\nSpatial Gap: {len(gap_df):,} grid 1x1 deg aktif 1963-{HIST_END} tapi kosong {RECENT_START}-{RECENT_END}")
print("\nDistribusi spatial gap per zona:")
print(f"{'Zona':<25} {'N Gap Grids':>12}")
print("-" * 42)
for z in ZONES:
    n = (gap_df["zone_name"] == z).sum()
    print(f"  {z:<23} {n:>12,}")

# ── Plot spatial gap map ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_facecolor("#d4e6f1")

ax.scatter(df["longitude"], df["latitude"], s=0.2, c="lightgray", alpha=0.15, rasterized=True)

for z in ZONES:
    sub = df_recent[df_recent["zone_name"] == z]
    ax.scatter(sub["longitude"], sub["latitude"], s=0.5, c=COLORS[z], alpha=0.3,
               label=f"{z} ({len(sub):,})", rasterized=True)

if len(gap_df) > 0:
    from matplotlib.patches import Rectangle
    for _, row in gap_df.iterrows():
        lat0 = row["lat_center"] - 0.5
        lon0 = row["lon_center"] - 0.5
        rect = Rectangle((lon0, lat0), 1, 1,
                          linewidth=0.3, edgecolor="red", facecolor="red", alpha=0.25)
        ax.add_patch(rect)
    gap_patch = mpatches.Patch(color="red", alpha=0.4, label=f"Seismic Gap ({len(gap_df)} grid)")

ax.set_xlim(90, 145)
ax.set_ylim(-12, 10)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Peta Seismic Gap Indonesia\nMerah = grid aktif 1963-{HIST_END} tapi kosong {RECENT_START}-{RECENT_END}",
             fontsize=11)
handles, labels = ax.get_legend_handles_labels()
if len(gap_df) > 0:
    handles.append(gap_patch)
    labels.append(f"Seismic Gap ({len(gap_df)} grid)")
ax.legend(handles=handles, labels=labels, loc="upper left", fontsize=8, ncol=2)
ax.grid(alpha=0.3)

plt.tight_layout()
out_path = OUTDIR / "seismic_gap_map.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n[OK] Disimpan: {out_path.name}")


## Cell 7 — Hazard Scoring & Radar Chart

In [ ]:
# ── Komponen Skor Bahaya ─────────────────────────────────────────────────────
# 1. b-value score (25%):   b rendah → bahaya tinggi → score = 1 - normalize(b)
# 2. recurrence score (25%): RI pendek → bahaya tinggi → score = 1 - normalize(RI)
# 3. gap area score (30%):   gap luas → bahaya tinggi → normalize(n_gap_grids)
# 4. mean_mag score (20%):   mag tinggi → bahaya tinggi → normalize(mean_mag)

WEIGHTS = {"b_score": 0.25, "ri_score": 0.25, "gap_score": 0.30, "mag_score": 0.20}

# Kumpulkan raw values
raw = {}
for z in ZONES:
    b  = gr_stats[z]["b"]
    ri = gr_stats[z]["recurrence_yr"]
    ng = (gap_df["zone_name"] == z).sum() if len(gap_df) > 0 else 0
    mm = df[df["zone_name"] == z]["mag"].mean()
    raw[z] = {"b": b if not np.isnan(b) else 1.0,
              "ri": ri if not np.isnan(ri) else 100.0,
              "gap": ng,
              "mean_mag": mm}

# Normalize ke [0,1] per komponen
def normalize_col(vals_dict, key, invert=False):
    vals = np.array([vals_dict[z][key] for z in ZONES], dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        norm = np.zeros(len(ZONES))
    else:
        norm = (vals - vmin) / (vmax - vmin)
    return (1 - norm) if invert else norm

b_norm   = normalize_col(raw, "b",        invert=True)   # b rendah = bahaya tinggi
ri_norm  = normalize_col(raw, "ri",       invert=True)   # RI pendek = bahaya tinggi
gap_norm = normalize_col(raw, "gap",      invert=False)  # gap besar = bahaya tinggi
mag_norm = normalize_col(raw, "mean_mag", invert=False)  # mag tinggi = bahaya tinggi

hazard_scores = {}
for i, z in enumerate(ZONES):
    comp = {"b_score": b_norm[i], "ri_score": ri_norm[i],
            "gap_score": gap_norm[i], "mag_score": mag_norm[i]}
    total = sum(comp[k] * WEIGHTS[k] for k in WEIGHTS)
    hazard_scores[z] = {**comp, "total_hazard": round(total, 4)}

# ── Radar Chart ──────────────────────────────────────────────────────────────
labels_radar = ["b-value\n(25%)", "Recurrence\n(25%)", "Gap Area\n(30%)", "Mean Mag\n(20%)"]
N = len(labels_radar)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close circle

fig = plt.figure(figsize=(14, 8))
fig.suptitle("Skor Bahaya Seismik per Zona (Radar Chart)", fontsize=13, fontweight="bold")

# Subplot 1: Radar
ax_radar = fig.add_subplot(121, polar=True)
for z in ZONES:
    h = hazard_scores[z]
    vals = [h["b_score"], h["ri_score"], h["gap_score"], h["mag_score"]]
    vals += vals[:1]
    ax_radar.plot(angles, vals, color=COLORS[z], lw=2, label=z)
    ax_radar.fill(angles, vals, color=COLORS[z], alpha=0.08)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(labels_radar, fontsize=9)
ax_radar.set_ylim(0, 1)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(["0.25","0.50","0.75","1.0"], fontsize=7)
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax_radar.set_title("Komponen Skor (0–1)", fontsize=10, pad=15)

# Subplot 2: Bar total skor
ax_bar = fig.add_subplot(122)
zones_sorted = sorted(ZONES, key=lambda z: hazard_scores[z]["total_hazard"], reverse=True)
total_vals   = [hazard_scores[z]["total_hazard"] for z in zones_sorted]
colors_bar   = [COLORS[z] for z in zones_sorted]
x = np.arange(len(zones_sorted))
bars = ax_bar.bar(x, total_vals, color=colors_bar, width=0.55, alpha=0.88, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, total_vals):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax_bar.set_xticks(x)
ax_bar.set_xticklabels([z.replace("Zona ","") for z in zones_sorted], rotation=15, ha="right", fontsize=9)
ax_bar.set_ylabel("Total Skor Bahaya (0–1)")
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title("Peringkat Skor Bahaya per Zona", fontsize=10)
ax_bar.axhline(np.mean(total_vals), color="gray", ls="--", lw=1, label=f"Mean = {np.mean(total_vals):.3f}")
ax_bar.legend(fontsize=8)
ax_bar.grid(axis="y", alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_path = OUTDIR / "radar_bahaya.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Disimpan: {out_path.name}")

# Tabel skor
print("\nSkor Bahaya Lengkap per Zona:")
print(f"{'Zona':<25} {'b-score':>9} {'RI-score':>9} {'Gap-score':>10} {'Mag-score':>10} {'TOTAL':>8}")
print("-" * 78)
for z in zones_sorted:
    h = hazard_scores[z]
    print(f"  {z:<23} {h['b_score']:>9.3f} {h['ri_score']:>9.3f} "
          f"{h['gap_score']:>10.3f} {h['mag_score']:>10.3f} {h['total_hazard']:>8.4f}")


## Cell 8 — Simpan Output CSV

In [ ]:
import re
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

WORKDIR = Path(".")
INPUT_CSV = WORKDIR / "indonesia_earthquakes_clustered.csv"
OUTDIR    = WORKDIR / "output_spatio_temporal"
OUTDIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_CSV)
df["time"] = pd.to_datetime(df["time"], format="mixed", errors="coerce")
df["year"] = df["time"].dt.year

ZONES = ["Zona Sumatera", "Zona Jawa-Bali-NTB", "Zona Sulawesi-NTT", "Zona Maluku", "Zona Papua"]

MC           = 4.0
M_RECUR      = 5.0
N_YEARS      = 62
RECENT_START = 2016
RECENT_END   = 2026

# ── Helper functions ─────────────────────────────────────────────────────────
def compute_bvalue_mle(mags, mc=MC):
    mags_above = mags[mags >= mc]
    n = len(mags_above)
    if n < 10:
        return np.nan, np.nan, np.nan, n
    mean_m = mags_above.mean()
    b = np.log10(np.e) / (mean_m - mc)
    a = np.log10(n / N_YEARS) + b * mc
    mag_bins   = np.arange(mc, mags_above.max() + 0.1, 0.2)
    n_cum      = np.array([(mags_above >= m).sum() for m in mag_bins], dtype=float)
    n_cum_pred = 10 ** (a - b * mag_bins)
    valid = n_cum > 0
    if valid.sum() > 2:
        log_obs  = np.log10(n_cum[valid])
        log_pred = np.log10(n_cum_pred[valid])
        ss_res = np.sum((log_obs - log_pred) ** 2)
        ss_tot = np.sum((log_obs - log_obs.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    else:
        r2 = np.nan
    return round(b, 4), round(a, 4), round(r2, 4), n

def recurrence_days(mags, b, a, m_target=M_RECUR):
    if np.isnan(b):
        return np.nan
    annual_rate = 10 ** (a - b * m_target)
    return round(365.25 / annual_rate, 1) if annual_rate > 0 else np.nan

def energy_joules(mags):
    return float(np.sum(10 ** (1.5 * mags + 4.8)))

def grid_to_latlon(gid):
    m = re.match(r"LAT(-?\d+)_LON(-?\d+)", gid)
    if not m:
        return np.nan, np.nan
    return int(m.group(1)) + 0.5, int(m.group(2)) + 0.5

def normalize_col(vals_dict, key, invert=False):
    vals = np.array([vals_dict[z][key] for z in ZONES], dtype=float)
    vmin, vmax = vals.min(), vals.max()
    norm = np.zeros(len(ZONES)) if vmax == vmin else (vals - vmin) / (vmax - vmin)
    return (1 - norm) if invert else norm

def hazard_category(score):
    if score >= 0.70: return "Sangat Tinggi"
    if score >= 0.55: return "Tinggi"
    if score >= 0.40: return "Sedang"
    return "Rendah"

# ── Spatial gap ──────────────────────────────────────────────────────────────
df_hist   = df[df["year"] <= RECENT_START - 1]
df_recent = df[(df["year"] >= RECENT_START) & (df["year"] <= RECENT_END)]
hist_grids   = set(df_hist["grid_id"].unique())
recent_grids = set(df_recent["grid_id"].unique())
spatial_gaps = hist_grids - recent_grids

gap_coords = [grid_to_latlon(g) for g in spatial_gaps]
gap_df = pd.DataFrame(gap_coords, columns=["lat_center", "lon_center"])
gap_df["grid_id"] = list(spatial_gaps)
grid_zone = (df_hist[["grid_id", "zone_name"]]
             .drop_duplicates(subset="grid_id", keep="first")
             .set_index("grid_id")["zone_name"])
gap_df["zone_name"] = gap_df["grid_id"].map(grid_zone)

# ── G-R stats per zona ───────────────────────────────────────────────────────
gr_stats = {}
for z in ZONES:
    mags     = df[df["zone_name"] == z]["mag"].dropna().values
    b, a, r2, n_above = compute_bvalue_mle(mags)
    ri_days  = recurrence_days(mags, b, a)
    e_total  = energy_joules(mags)
    mean_mag = round(float(mags.mean()), 4)
    max_mag  = round(float(mags.max()),  4)
    ng       = int((gap_df["zone_name"] == z).sum())
    gr_stats[z] = {
        "b": b, "a": a, "r2": r2, "n_above_mc": n_above,
        "recurrence_days": ri_days, "energy_total": e_total,
        "mean_mag": mean_mag, "max_mag": max_mag,
        "gap_area_deg2": ng, "n_events": int(len(df[df["zone_name"] == z])),
    }

# ── Hazard scoring ────────────────────────────────────────────────────────────
raw = {}
for z in ZONES:
    raw[z] = {
        "b":        gr_stats[z]["b"]           if not np.isnan(gr_stats[z]["b"])           else 1.0,
        "ri":       gr_stats[z]["recurrence_days"] if not np.isnan(gr_stats[z]["recurrence_days"]) else 36500.0,
        "gap":      gr_stats[z]["gap_area_deg2"],
        "mean_mag": gr_stats[z]["mean_mag"],
    }

b_norm   = normalize_col(raw, "b",        invert=True)
ri_norm  = normalize_col(raw, "ri",       invert=True)
gap_norm = normalize_col(raw, "gap",      invert=False)
mag_norm = normalize_col(raw, "mean_mag", invert=False)

hazard_scores = {}
for i, z in enumerate(ZONES):
    total = b_norm[i]*0.25 + ri_norm[i]*0.25 + gap_norm[i]*0.30 + mag_norm[i]*0.20
    hazard_scores[z] = round(total, 4)

# ════════════════════════════════════════════════════════════════════════════
# FILE 1: spatio_temporal_summary.csv
# ════════════════════════════════════════════════════════════════════════════
rows = []
for z in ZONES:
    s  = gr_stats[z]
    hs = hazard_scores[z]
    rows.append({
        "zone_name":                z,
        "n_event":                  s["n_events"],
        "b_value":                  s["b"],
        "a_value":                  s["a"],
        "r2":                       s["r2"],
        "recurrence_interval_days": s["recurrence_days"],
        "mean_mag":                 s["mean_mag"],
        "max_mag":                  s["max_mag"],
        "energy_total":             round(s["energy_total"], 3),
        "gap_area_deg2":            s["gap_area_deg2"],
        "hazard_score":             hs,
        "hazard_category":          hazard_category(hs),
    })

summary_df = pd.DataFrame(rows)
out1 = OUTDIR / "spatio_temporal_summary.csv"
summary_df.to_csv(out1, index=False, encoding="utf-8")
print(f"[OK] {out1.name}  ({len(summary_df)} baris x {len(summary_df.columns)} kolom)")
print()
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
print(summary_df.to_string(index=False))

# ════════════════════════════════════════════════════════════════════════════
# FILE 2: seismic_gap_zones.csv
# ════════════════════════════════════════════════════════════════════════════
gap_rows = []

yearly_all = (
    df.groupby(["year", "zone_name"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=ZONES, fill_value=0)
)
yearly_all = yearly_all.loc[(yearly_all.index >= 1963) & (yearly_all.index <= 2024)]

for z in ZONES:
    series    = yearly_all[z]
    mu, sigma = series.mean(), series.std()
    threshold = mu - 1.5 * sigma
    gap_years = series[series < threshold].index.tolist()
    if gap_years:
        last_active = int(series[series > 0].index.max()) if (series > 0).any() else "N/A"
        gap_rows.append({
            "zone_name":       z,
            "gap_type":        "temporal",
            "location":        f"Seluruh zona {z}",
            "area":            "N/A",
            "last_event_year": last_active,
            "description": (
                f"{len(gap_years)} tahun di bawah threshold "
                f"({threshold:.1f} event/yr): {gap_years[:5]}"
                + ("..." if len(gap_years) > 5 else "")
            ),
        })

if len(gap_df) > 0:
    grid_hist_count = df_hist.groupby("grid_id").size().rename("hist_count")
    gap_df2 = gap_df.copy().join(grid_hist_count, on="grid_id")
    gap_df2 = gap_df2.sort_values("hist_count", ascending=False)
    for z in ZONES:
        sub_gap = gap_df2[gap_df2["zone_name"] == z].head(10)
        for _, row in sub_gap.iterrows():
            hist_events = df_hist[df_hist["grid_id"] == row["grid_id"]]
            last_yr     = int(hist_events["year"].max()) if len(hist_events) > 0 else "N/A"
            gap_rows.append({
                "zone_name":       z,
                "gap_type":        "spatial",
                "location":        row["grid_id"],
                "area":            "1x1 deg (~12000 km2)",
                "last_event_year": last_yr,
                "description": (
                    f"Aktif s/d {last_yr}, nol event "
                    f"{RECENT_START}-{RECENT_END} "
                    f"(hist: {int(row.get('hist_count', 0))} events)"
                ),
            })

gap_zones_df = pd.DataFrame(gap_rows)
out2 = OUTDIR / "seismic_gap_zones.csv"
gap_zones_df.to_csv(out2, index=False, encoding="utf-8")
print(f"\n[OK] {out2.name}  ({len(gap_zones_df)} baris x {len(gap_zones_df.columns)} kolom)")
print()
print(gap_zones_df.head(20).to_string(index=False))
if len(gap_zones_df) > 20:
    print(f"  ... ({len(gap_zones_df) - 20} baris lagi tidak ditampilkan)")

# ════════════════════════════════════════════════════════════════════════════
# KONFIRMASI SEMUA OUTPUT
# ════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*62}")
print("KONFIRMASI OUTPUT -- output_spatio_temporal/")
print(f"{'='*62}")
expected = [
    "temporal_trend_per_zona.png",
    "seasonal_decompose_per_zona.png",
    "spatial_per_zona.png",
    "bvalue_per_zona.png",
    "seismic_gap_map.png",
    "radar_bahaya.png",
    "spatio_temporal_summary.csv",
    "seismic_gap_zones.csv",
]
all_ok = True
for fname in expected:
    fpath = OUTDIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f"  [OK]      {fname:<42} {size_kb:>7.1f} KB")
    else:
        print(f"  [MISSING] {fname}")
        all_ok = False
print(f"{'='*62}")
if all_ok:
    print("SEMUA OUTPUT LENGKAP")
else:
    print("PERHATIAN: PNG dari Cell 2-7 harus dijalankan dulu di notebook")
